In [62]:
import json

In [63]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

def build_formula(criteria, parent_key=''):
    if isinstance(criteria, dict):
        formula_parts = []
        for key, value in criteria.items():
            if key in ['AND', 'OR', 'NOT OR']:
                if key == 'NOT OR':
                    formula_part = f"NOT({build_formula(value, key)})"
                else:
                    formula_part = f"({build_formula(value, key)})"
                formula_parts.append(formula_part)
            else:
                formula_parts.append(build_formula(value, key))
        connector = ' OR ' if parent_key == 'OR' else ' AND '
        return connector.join(formula_parts)
    elif isinstance(criteria, str):
        return parent_key
    else:
        return ''

In [69]:
data = load_json("set_2.json")
build_formula(data)

'IC1.1 AND IC1.2 AND IC2.1 AND IC2.2 AND IC2.3 AND IC3 AND EC1.1 AND EC1.2 AND EC1.3 AND EC2.1 AND EC2.2 AND EC3.1 AND EC3.2 AND EC4 AND EC5 AND EC6'

In [70]:
data

{'IC AND': {'IC1 OR': {'IC1.1': 'Overweight subjects [according to body mass index (BMI)]',
   'IC1.2': 'Obese subjects [according to body mass index (BMI)]'},
  'IC2 AND': {'IC2.1': 'Fasting plasma glucose value between 100 and 125 mg/dl',
   'IC2.1 OR': {'IC2.2': 'Impaired fasting glucose confirmed with oral glucose tolerance test (OGTT)',
    'IC2.3': 'Impaired glucose tolerance confirmed with oral glucose tolerance test (OGTT)'}},
  'IC3': 'Total cholesterol values ≥ 200 mg/dl'},
 'EC NOT OR': {'EC1 OR': {'EC1.1': 'Patients with neoplastic diseases',
   'EC1.2': 'Patients with liver diseases',
   'EC1.3': 'Patients with renal failure'},
  'EC2 OR': {'EC2.1': 'Patients with type 1 diabetes mellitus',
   'EC2.2': 'Patients with type 2 diabetes mellitus'},
  'EC3 OR': {'EC3.1': 'Pregnant women', 'EC3.2': 'Breastfeeding women'},
  'EC4': 'Hypersensitivity to any of the ingredients',
  'EC5': 'Therapy with lipid-lowering drugs',
  'EC6': 'Use of products containing red yeast rice'}}

In [71]:
def generate_latex_formula(criteria, parent_key='', is_root=True):
    formula_parts = []
    for key, value in criteria.items():
        # Entferne IC und EC Präfixe von logischen Operatoren und speichere den reinen Operator
        cleaned_key = key.replace("IC ", "").replace("EC ", "")

        # Wenn der Wert ein weiteres Dictionary ist, gehe rekursiv vor
        if isinstance(value, dict):
            formula_part = generate_latex_formula(value, cleaned_key, False)
            # Für "NOT OR" wird eine Negation hinzugefügt
            if cleaned_key == 'NOT OR':
                formula_parts.append(f"\\neg ({formula_part})")
            else:
                formula_parts.append(formula_part)
        else:
            # Direkte Kriterien (nicht logisch) einfach hinzufügen
            formula_parts.append(key)

    # Wähle den korrekten logischen Operator basierend auf dem Schlüssel
    connector = ' \\lor ' if parent_key == 'OR' else ' \\wedge '
    joined_parts = connector.join(formula_parts).strip()

    # Um unnötige Klammern an der Wurzelebene zu vermeiden und die Ausgabe klar zu halten
    return f"({joined_parts})" if not is_root and formula_parts else joined_parts

In [72]:
generate_latex_formula(data)

'((IC1.1 \\wedge IC1.2) \\wedge (IC2.1 \\wedge (IC2.2 \\wedge IC2.3)) \\wedge IC3) \\wedge \\neg (((EC1.1 \\wedge EC1.2 \\wedge EC1.3) \\wedge (EC2.1 \\wedge EC2.2) \\wedge (EC3.1 \\wedge EC3.2) \\wedge EC4 \\wedge EC5 \\wedge EC6))'